In [29]:
import duckdb
from dotenv import load_dotenv
import os
import logging


In [30]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    datefmt= '%Y-%m-%d %H:%M:%S')

logger = logging.getLogger(__name__)
logging.info('info')
logging.debug('debug')
logging.warning('warning')
logging.error('error')
logging.critical("critical")

2026-08-17 12:54:54 - INFO - info
2026-08-17 12:54:54 - WARNING - warning
2026-08-17 12:54:54 - ERROR - error
2026-08-17 12:54:54 - CRITICAL - critical


In [31]:

con = duckdb.connect()
table = "cdc_pgdb.diabetes_ind"
SCHEMA = os.getenv("CDC_DB_SCHEMA")
con.execute(f"ATTACH '{os.getenv("CDC_CONNECTION_STRING")}' AS cdc_pgdb (TYPE postgres, SCHEMA {SCHEMA});")
logger.info("Data Exploration and Quality Check...\n")


2026-08-17 12:54:54 - INFO - Data Exploration and Quality Check...



In [ ]:
logger.info(f"Counting Total Numbers of Rows in our Database:")
total_values = con.execute(f"SELECT COUNT(*) FROM {table}").fetchdf()
total_values



2026-08-17 12:54:54 - INFO - Counting Total Numbers of Rows in our Database:


,count_star()
0,45144


In [33]:
logger.info(f"\nDistinct years in our year column:")
distinct_year = con.execute(f"SELECT DISTINCT year FROM {table} ORDER BY year ASC").fetchdf()
distinct_year


2026-08-17 12:54:54 - INFO - 
Distinct years in our year column:


,year
0,2000
1,2001
2,2002
3,2003
4,2004
5,2005
6,2006
7,2007
8,2008
9,2009


In [34]:
logger.info(f"\n Distinct indicators for diabetes:")
distinct_indicators = con.execute(f"SELECT DISTINCT indicator FROM {table}").fetchdf()
distinct_indicators


2026-08-17 12:54:54 - INFO - 
 Distinct indicators for diabetes:


,indicator
0,Visual Impairment
1,Current Tobacco Use
2,Prevalence of Cardiovascular Disease
3,Body Mass Index (BMI)
4,Hospitalization (any-listed)
5,Hospitalization for Heart Failure
6,Pre-pregnancy Diabetes Mellitus (Female Only)
7,Hospitalization for Diabetic Ketoacidosis (DKA)
8,Received Pneumococcal Vaccination
9,Hospitalization (first-listed)


In [35]:
logger.info(f"\n Population by group:")
population_by_group = con.execute(f"SELECT population, COUNT(*) FROM {table} GROUP BY population").fetchdf()
population_by_group

2026-08-17 12:54:54 - INFO - 
 Population by group:


,population,count_star()
0,Adults with Diabetes Aged 18+ Years,26372
1,All Ages,396
2,Deliveries among Females Aged 15-44 Years,1242
3,Adults with Diabetes Aged 35+ years,2700
4,All Ages with Diabetes,638
5,Adults without Diabetes Aged 18+ Years,2224
6,Adults Aged 18+ Years,9320
7,Children & Adolescents Aged 0-19 Years,960
8,Adults with Diabetes Aged 60+ years,1292


In [36]:
logger.info(f"\nMissing Estimates:")
missing_values = con.execute(f"SELECT COUNT(*) AS total, COUNT(estimate) AS has_estimates, COUNT(*) - COUNT(estimate) as missing_estimates FROM {table}").fetchdf()
missing_values

2026-08-17 12:54:54 - INFO - 
Missing Estimates:


,total,has_estimates,missing_estimates
0,45144,39135,6009


In [37]:

logger.info(f"Combination of age, sex, education, and race:")
combination_records = con.execute(f"Select age, sex, education, race, COUNT(*) FROM {table} GROUP BY age, sex, education, race ORDER BY COUNT(*) DESC LIMIT 20").fetchdf()
combination_records

2026-08-17 12:54:54 - INFO - Combination of age, sex, education, and race:


,age,sex,education,race,count_star()
0,Crude,All,All,All,2577
1,Crude,Male,All,All,2485
2,Crude,Female,All,All,2485
3,75+,All,All,All,2405
4,Crude,All,All,Non-Hispanic Black,2385
5,Crude,All,All,Non-Hispanic White,2385
6,Crude,All,All,Hispanic,2385
7,65-74,All,All,All,2329
8,45-64,All,All,All,2179
9,18-44,All,All,All,2113
